In [1]:
def build_embedding_text(article):
    parts = [
        f"Chương {article['chapter_id']}: {article['chapter_title']}"
    ]

    if article["section_id"] != 0:
        parts.append(
            f"Mục {article['section_id']}: {article['section_title']}"
        )

    parts.append(
        f"Điều {article['article_id']}: {article['article_title']}"
    )

    parts.append(article["content"])

    return "passage: " + "\n\n".join(parts)

In [2]:
import json
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

with open(
    "articles.json",
    "r",
    encoding="utf-8"
) as f:
    articles = json.load(f)

texts = [build_embedding_text(a) for a in articles]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=16
)

print(embeddings.shape)

c:\Users\vietl\miniconda3\envs\labor\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 14/14 [00:47<00:00,  3.41s/it]

(220, 768)


In [3]:
import numpy as np

def search(query, top_k=5):

    query_embedding = model.encode(
        f"query: {query}",
        normalize_embeddings=True
    )

    scores = embeddings @ query_embedding

    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_idx:
        results.append({
            "score": float(scores[idx]),
            "id": articles[idx]["id"],
            "title": articles[idx]["article_title"]
        })

    return results

In [11]:
query_text = f"query: {'Sau khi ký hợp động lao động tập thể, người chủ phải làm gì'}"
results = search(
    query_text
)

for r in results:
    print(r)

{'score': 0.8677780628204346, 'id': 'C3M5D54', 'title': 'Doanh nghiệp cho thuê lại lao động'}
{'score': 0.8657588362693787, 'id': 'C5M3D82', 'title': 'Sửa đổi, bổ sung thỏa ước lao động tập thể'}
{'score': 0.8647124171257019, 'id': 'C5M3D76', 'title': 'Lấy ý kiến và ký kết thỏa ước lao động tập thể'}
{'score': 0.8636503219604492, 'id': 'C5M3D85', 'title': 'Gia nhập và rút khỏi thỏa ước lao động tập thể ngành hoặc thỏa ước lao động tập thể có'}
{'score': 0.863160252571106, 'id': 'C5M3D77', 'title': 'Gửi thỏa ước lao động tập thể'}
